## 1. Streszczenie

Celem projektu jest wytrenowanie dwóch modeli, generujących muzykę, korzytsając z bibliotek torch. Do trenowania modeli LSTM oraz transformerowego użyta została oprawa muzyczna gier DOOM oraz DOOM 2, fanowskiego final DOOM, Duke Nukem 3D, Wolfenstein 3D, Blake Stone: Aliens of Gold, Xenophage: Alien Bloodsport oraz DemonStar zapisanych w formacie mid. Odrzucono powtarzające się i odstające od reszty utwory a następnie utworzeno na ich podstawie słownik tokenów. Trening odbywał sie w wielu wariantach, aby umożliwić modelom jak najlepsze odwzorowanie stylu kompozytora Roberta Prince'a.

## 2. Wstęp / wprowadzenie

Generowanie muzyki z użyciem sieci neuronowych stanowi wymagające zadanie z zakresu modelowania sekwencji, ponieważ model musi uchwycić zarówno lokalne zależności między kolejnymi zdarzeniami, jak i dłuższą strukturę rytmiczną oraz motywiczną utworu. W przypadku muzyki symbolicznej problem ten rozpatruje się najczęściej na poziomie zdarzeń MIDI, co pozwala operować na reprezentacji nut, instrumentów i przesunięć czasowych zamiast na surowym sygnale audio.

W literaturze stosuje się w tym celu różne architektury autoregresyjne, w szczególności modele rekurencyjne typu LSTM oraz nowsze modele oparte na mechanizmie self-attention, takie jak Transformer. Pierwsze z nich dobrze modelują lokalną ciągłość sekwencji, natomiast drugie lepiej radzą sobie z uchwyceniem zależności długoterminowych i bardziej złożonej struktury muzycznej.

W projekcie rozważono oba podejścia w zadaniu generowania muzyki w stylistyce ścieżki dźwiękowej gier *Doom* i *Doom II* autorstwa Bobby'ego Prince'a. Wybrany zbiór danych jest stosunkowo mały, ale jednocześnie spójny stylistycznie i dostępny w formacie MIDI, co czyni go dobrym materiałem do eksperymentów z modelowaniem muzyki symbolicznej.

## 3. Opis kontekstu i celu

Projekt wpisuje się w obszar generowania muzyki symbolicznej, w którym utwór reprezentowany jest jako sekwencja zdarzeń MIDI przewidywanych na podstawie wcześniejszego kontekstu. Jest to zadanie zbliżone formalnie do modelowania języka, jednak w praktyce wymaga uchwycenia nie tylko kolejności zdarzeń, ale również regularności rytmicznej, powtarzalności motywów i spójności w dłuższych odcinkach sekwencji.

Istotnym kontekstem projektu jest porównanie dwóch klas architektur używanych w generowaniu muzyki: modeli rekurencyjnych LSTM oraz modeli opartych na mechanizmie uwagi, takich jak Transformer. LSTM stanowi klasyczne podejście do modelowania sekwencji i dobrze nadaje się do uchwycenia zależności lokalnych, natomiast Transformer lepiej radzi sobie z modelowaniem relacji długoterminowych dzięki bezpośredniemu dostępowi do całego kontekstu wejściowego.

Jako materiał treningowy wybrano utwory z gier *Doom* i *Doom II* autorstwa Bobby'ego Prince’a. Zbiór ten jest relatywnie mały, lecz jednorodny stylistycznie i dostępny w formacie MIDI.

Głównym celem projektu było zbudowanie i przetestowanie modeli zdolnych do generowania nowych sekwencji MIDI utrzymanych w stylistyce oryginalnej muzyki z serii *Doom*. Celem dodatkowym było sprawdzenie, jak model LSTM i model Transformer zachowują się w warunkach małego zbioru danych oraz które cechy generowanej muzyki każdy z nich odwzorowuje lepiej.

Od strony praktycznej projekt obejmował cały proces eksperymentalny: przygotowanie danych, tokenizację zdarzeń, implementację i trening modeli, a następnie generowanie i odsłuch otrzymanych sekwencji. Ocena jakości miała charakter zarówno ilościowy, na podstawie funkcji straty, jak i jakościowy, przez analizę odsłuchową wygenerowanej muzyki.

## 5. Opis danych

W projekcie wykorzystano zbiór plików MIDI zawierających muzykę z gier *Doom* i *Doom II*. Dane zostały pobrane z dwóch archiwów: [`doommusic.zip`](https://archive.org/download/doommusic) oraz [`doom2music.zip`](https://archive.org/download/doom2music), a po ich rozpakowaniu uzyskano łącznie 67 surowych plików MIDI.

Wybrany materiał ma charakter symboliczny, co oznacza, że każdy utwór zapisany jest jako zbiór zdarzeń muzycznych, takich jak nuty, instrumenty czy informacje czasowe, a nie jako sygnał audio. Taka forma reprezentacji jest wygodna w zadaniu generowania muzyki, ponieważ umożliwia analizę struktury utworu bez konieczności pracy na surowej fali dźwiękowej.

### 5.1 Czyszczenie danych

W ramach wstępnej analizy dla każdego pliku wyznaczono podstawowe statystyki, obejmujące między innymi czas trwania, liczbę instrumentów, liczbę nut, średnie tempo oraz informację o obecności ścieżki perkusyjnej. Na tej podstawie przeprowadzono czyszczenie zbioru: wykonano deduplikację po liczbie nut, odrzucono utwory krótsze niż 60 sekund oraz usunięto plik `dbunny.mid`, który stylistycznie odbiegał od pozostałych utworów. W rezultacie końcowy zbiór zawierał 42 utwory przeznaczone do dalszego przetwarzania.

Po oczyszczeniu danych wykonano analizę eksploracyjną zbioru. Rozkład liczby nut w utworach pozwala ocenić zróżnicowanie gęstości muzycznej pomiędzy ścieżkami, natomiast rozkład długości utworów pokazuje, że zbiór obejmuje zarówno krótsze kompozycje, jak i wyraźnie dłuższe sekwencje. Uzupełnieniem tej analizy jest wykres udziału utworów zawierających ścieżkę perkusyjną, co ma znaczenie z punktu widzenia rytmicznego charakteru materiału.

Na podstawie przeprowadzonej analizy można stwierdzić, że zbiór danych jest niewielki liczebnie, ale zróżnicowany pod względem długości utworów i liczby nut. Oznacza to, że modele uczone na tych danych mają kontakt zarówno z krótszymi i prostszymi sekwencjami, jak i z dłuższymi, bardziej gęstymi przypadkami.

![Rozkład liczby nut w utworach.](Note_count_distribution.png)
**Rysunek 1.** Rozkład liczby nut w utworach.

![Rozkład długości ścieżek MIDI w zbiorze danych.](Track_len_dirtribution.png)
**Rysunek 2.** Rozkład długości ścieżek MIDI w zbiorze danych.

Dodatkowo zdecydowana większość utworów zawiera ścieżkę perkusyjną, dokładnie 92,86% zbioru. Miało to wpływ na przyjętą reprezentację danych, ponieważ w tokenizacji uwzględniono osobny typ tokenów dla zdarzeń perkusyjnych. Jest to uzasadnione zarówno częstą obecnością perkusji w zbiorze, jak i jej istotną rolą w stylistyce muzyki z serii *Doom*.

![Udział utworów zawierających ścieżkę perkusyjną.](contain_drums.png)
**Rysunek 3.** Udział utworów zawierających ścieżkę perkusyjną.

### 5.2 Reprezentacja i tokenizacja danych

Aby umożliwić uczenie modeli sekwencyjnych, każdy utwór MIDI został przekształcony do reprezentacji event-based, inspirowanej podejściami stosowanymi w modelach generowania muzyki symbolicznej. Zamiast kodować muzykę jako sztywną siatkę pianorollową, zastosowano sekwencję tokenów opisujących konkretne zdarzenia muzyczne zachodzące w czasie. Takie podejście lepiej zachowuje strukturę utworu i pozwala reprezentować rytm, jak i instrumentację.

W implementacji przyjęto parametr `STEPS_PER_BEAT = 16`, co oznacza próbkowanie czasu do szesnastu kroków na uderzenie. Dodatkowo zastosowano `MAX_SHIFT = 64`, dzięki czemu pojedynczy token przesunięcia czasu mógł reprezentować maksymalnie 64 kroki czasowe. Pozwalało to ograniczyć długość sekwencji bez utraty podstawowej informacji rytmicznej.

Zdefiniowany słownik składał się z następujących typów tokenów:
- `PAD` – token do wyrównywania długości sekwencji,
- `BOS` – znacznik początku sekwencji,
- `EOS` – znacznik końca sekwencji,
- `NOTE_ON_x` – rozpoczęcie nuty o wysokości `x`,
- `NOTE_OFF_x` – zakończenie nuty o wysokości `x`,
- `DRUM_x` – zdarzenie perkusyjne o wysokości `x`,
- `PROGRAM_x` – zmiana instrumentu na program `x`,
- `SHIFT_x` – przesunięcie czasu o `x` kroków.

Dla każdego zdarzenia melodycznego wykorzystywano osobne tokeny `NOTE_ON` i `NOTE_OFF`, co pozwalało jawnie reprezentować czas trwania nut. Zdarzenia perkusyjne kodowano inaczej, za pomocą pojedynczego tokenu `DRUM_x`, bez osobnego `NOTE_OFF`, ponieważ perkusja ma w praktyce charakter krótkiego, jednorazowego uderzenia. Przed wystąpieniem nuty emitowany był również token `PROGRAM_x`, dzięki czemu model otrzymywał informację o aktualnym instrumencie.

Ruch w czasie nie był kodowany bezwzględnymi pozycjami, lecz za pomocą tokenów `SHIFT_x`. Oznacza to, że jeśli pomiędzy kolejnymi zdarzeniami występowała przerwa, była ona zapisywana jako odpowiednie przesunięcie czasu.

Wysokości dźwięków oraz programy instrumentów były kodowane w zakresie standardu MIDI od 0 do 127. Dla każdej możliwej wartości wysokości zdefiniowano osobne tokeny `NOTE_ON`, `NOTE_OFF` oraz `DRUM`, a dla każdego programu instrumentu osobny token `PROGRAM`. Uzupełnieniem były tokeny `SHIFT_1` do `SHIFT_64` oraz tokeny specjalne. Łączny rozmiar słownika wyniósł 579 tokenów.

W obecnej wersji projektu nie implementowano osobnych tokenów dynamiki. Prędkość nuty została ustawiona na stałą wartość `velocity = 100`, co upraszcza reprezentację, ale jednocześnie ogranicza ekspresję generowanej muzyki.

Oto przykładowa piosenka z datasetu:


<audio controls>
  <source src="../../output/example/d_e1m1.wav" type="audio/mp4">
  Twoja przeglądarka nie obsługuje odtwarzacza audio.
</audio>

## 6. Opis metod

W projekcie problem generowania muzyki potraktowano jako zadanie autoregresyjnego modelowania sekwencji. Oznacza to, że model na podstawie wcześniej występujących tokenów przewiduje kolejny element sekwencji, a następnie proces ten jest powtarzany aż do uzyskania pełnego utworu. Wejściem dla modeli były sekwencje tokenów opisujących zdarzenia MIDI, takie jak rozpoczęcie i zakończenie nuty, zmiana instrumentu, zdarzenia perkusyjne oraz przesunięcia czasowe.

### 6.1 LSTM

Pierwszą zastosowaną metodą był model LSTM (Long Short-Term Memory), czyli odmiana rekurencyjnej sieci neuronowej zaprojektowanej do pracy z danymi sekwencyjnymi. LSTM jest rodzajem  zaawansowanej sztucznej sieci neuronowej, która potrafi zapamiętywać informacje o wcześniejszych elemenatach sekwencji. Dzięki temu jest w stanie uwzględnić kontekst historyczny podczas przewidywań. Doskonale sprawdzaja sie przy przetwarzaniu języka naturalnego, rozpoznawaniu mowy, czy generowaniu tekstu i muzyki. Podczas gdy tradycyjne sieci neuronowe mają problem z pamiętaniem zdarzeń, które miały miejsce na początku długiej sekwencji (tzw. problem zanikającego gradientu). LSTM rozwiązuje ten problem za pomocą specjalnych "bramek" w swoich komórkach pamięci: 
- **Bramka zapominająca**: decyduje, które informacje z przeszłości są zbędne i należy je odrzucić
- **Bramka wejściowa**: decyduje, które nowe informacje są ważne i zostaną dodane do pamięci
- **Bramka wyjściowa**: decyduje, jakie informacje zostaną przekazane dalej jako wynik działania sieci.

### 6.2 Model transformerowy

Drugą zastosowaną metodą był model Transformer w wariancie decoder-only. W przeciwieństwie do LSTM nie przetwarza on sekwencji wyłącznie krok po kroku poprzez stan ukryty, lecz wykorzystuje mechanizm self-attention, który pozwala analizować zależności pomiędzy elementami całego kontekstu wejściowego. Transformery umożliwiają przetwarzanie równoległe, dzięki czemu mogą być trenowane na gigantycznych zbiorach danych przy użyciu nowoczesnych procesorów graficznych. Transformery wykorzystywane są w najpopularniejszych modelach językowych, takich jak GPT, Claude czy Gemini, a także w zaawansowanych systemach generujących muzykę i obrazy.

### 6.3 Generowanie sekwencji

Po wytrenowaniu obu modeli możliwe było generowanie nowych sekwencji muzycznych. Proces generowania polegał na podaniu modelowi początkowego fragmentu sekwencji, a następnie iteracyjnym przewidywaniu kolejnych tokenów. Otrzymana sekwencja tokenów była następnie dekodowana z powrotem do formatu MIDI, co umożliwiało odsłuch i ocenę wygenerowanego materiału muzycznego.

## 7. Opis wdrożenia metod / modelu

### 7.1 Przygotowanie danych wejściowych

W obu implementacjach wykorzystano środowisko PyTorch oraz dane zapisane w plikach `sequences.pkl` i `vocab.json`, zawierających odpowiednio sekwencje tokenów oraz słownik mapujący tokeny na identyfikatory liczbowe. Wejściem dla modeli były sekwencje identyfikatorów tokenów reprezentujących zdarzenia muzyczne, a rozmiar słownika wynosił 579 tokenów. W finalnym pipeline użytym w notebooku transformera zbiór obejmował 42 sekwencje o łącznej długości 382 035 tokenów.

Do budowy próbek treningowych wykorzystano klasę `DoomDataset`, która wycinała z pełnych sekwencji krótsze fragmenty wejściowe o zadanej długości kontekstu i kroku przesuwania okna. W modelu LSTM zastosowano długość kontekstu równą `256` tokenów oraz `STRIDE = 16`, natomiast w modelu transformerowym długość kontekstu zwiększono do 1024 tokenów przy tym `STRIDE = 64`. W implementacji LSTM jak i transformera włączono augmentację danych przez transpozycję z parametrem `MAX_TRANSPOSE = 12`.

### 7.2 Implementacja modelu LSTM

Model LSTM został zaimplementowany jako sieć z warstwą embeddingów o wymiarze 256, trzema warstwami LSTM, rozmiarem stanu ukrytego 512 oraz dropoutem równym 0.2 w części rekurencyjnej. Za blokiem LSTM zastosowano warstwę normalizacji `LayerNorm`, a następnie liniową warstwę wyjściową rzutującą reprezentację ukrytą na przestrzeń 579 tokenów słownika.

Uczenie modelu prowadzono z `BATCH_SIZE = 128` przez 25 epok. Funkcją straty była `CrossEntropyLoss`, a do optymalizacji wykorzystano algorytm Adam z parametrami `lr = 0.001` oraz `weight_decay = 0.01`. Dodatkowo zastosowano przycinanie gradientów do wartości `max_norm = 1.0`, co stabilizowało proces uczenia. W notebooku wygenerowano 5852 próbek treningowych, a pojedyncza paczka danych miała kształt `X: [128, 256]` oraz `y: [128, 256]`. Po zakończeniu treningu wagi modelu zapisywano do pliku `lstm_music_model.pth`.

### 7.3 Implementacja modelu transformerowego

Model transformerowy został zaimplementowany w wariancie decoder-only jako własna architektura złożona z warstwy osadzeń tokenów, sinusoidalnego kodowania pozycyjnego, maskowanej wielogłowej samouwagi, bloków feed-forward, warstw normalizacji oraz końcowej warstwy `lm_head` generującej logity dla całego słownika. W implementacji zastosowano parametry `d_model = 256`, `n_heads = 8`, `n_layers = 4`, `d_ff = 1024` oraz `dropout = 0.3`. Łączna liczba parametrów modelu wynosiła ~ 3.4 miliona.

W modelu zastosowano maskę przyczynową tworzoną z dolnotrójkątnej macierzy, dzięki czemu na każdej pozycji model mógł korzystać wyłącznie z wcześniejszych tokenów sekwencji. Przed wejściem do stosu dekoderów embeddingi tokenów były sumowane z kodowaniem pozycyjnym i poddawane normalizacji warstwowej. Taka konstrukcja pozwalała trenować model w trybie autoregresyjnym, przy zachowaniu porządku czasowego.

### 7.4 Proces treningu

W przypadku transformera dane podzielono na część treningową i walidacyjną przy `VAL_RATIO = 0.1` dzielącego nie tokeny, tylko utwory. Do walidacji trafiły cztery utwory: `d_dead.mid`, `d_e2m1.mid`, `d_e2m2.mid` oraz `d_victor.mid`, a zbiór treningowy obejmował 38 sekwencji o łącznej długości 350 558 tokenów, podczas gdy walidacyjny 4 sekwencje o długości 31 477 tokenów. Dla transformera użyto `BATCH_SIZE = 32`, funkcji straty `CrossEntropyLoss(ignore_index=PAD_ID)` oraz optymalizatora AdamW z parametrami `lr = 3e-4` i `weight_decay = 0.05`. Dodatkowo zastosowano uczenie mieszanej precyzji przy użyciu `autocast` i `GradScaler`.

Trening transformera zaplanowano na 60 epok, z zapisem checkpointu co 5 epok oraz mechanizmem early stopping, z którego jednak ostatecznie zrezygnowano. Przy early stopping model osiągał najniższe `val_loss` w przeciągu pierwszych 4 epok, co nie dawało zadowalających rezultatów. Najlepszy model zapisywano w pliku `transformer_best.pt`, a dodatkowe checkpointy tworzono również okresowo w plikach `transformer_epXXX.pt`.

W notebooku LSTM implementacja obejmowała trening na całym zbiorze treningowym oraz zapis końcowych wag modelu, bez osobno pokazanej procedury walidacyjnej i checkpointowania analogicznej do tej zastosowanej w transformerze.

### 7.5 Generowanie utworów

W notebooku transformera zaimplementowano pełny mechanizm generowania nowych sekwencji muzycznych. Generowanie rozpoczynało się od tokenu `BOS` albo od zadanego ziarna sekwencji, po czym model iteracyjnie przewidywał kolejne tokeny aż do osiągnięcia tokenu `EOS` lub limitu długości. Na każdym kroku wykorzystywano jedynie ostatnie 1024 tokeny jako aktualny kontekst wejściowy.

Podczas próbkowania zastosowano temperaturę, ograniczenie `top_k = 40` oraz `repetition_penalty = 1.05`, nakładane na 32 ostatnio wygenerowanye tokeny. Zaimplementowano również wariant generowania warunkowanego, w którym jako ziarno podawano pierwsze 100 tokenów istniejącej sekwencji charakterystycznego utworu `d_e1m1`.

Wygenerowane identyfikatory tokenów były następnie mapowane z powrotem na zdarzenia muzyczne i zapisywane do plików MIDI. Przy dekodowaniu przyjęto `STEPS_PER_BEAT = 16` oraz standardowe tempo `bpm = 95`. Zapisane przykłady różniły się długością i liczbą nut, co wskazuje, że model generował sekwencje o zauważalnie zróżnicowanej strukturze.

## 8. Wyniki 

### 8.1. Model LSTM

Model LSTM był w stanie generować sekwencje muzyczne zachowujące część cech charakterystycznych dla stylistyki ścieżki dźwiękowej serii *Doom*. Wygenerowane utwory zawierały rozpoznawalne zależności rytmiczne i melodyczne, jednak ich struktura bywała mniej spójna na dłuższych odcinkach. W praktyce jakość wyników uzyskiwanych przez LSTM w dużym stopniu zależała od doboru parametrów treningu.

Zaletą modelu LSTM był krótszy i prostszy proces trenowania. Dzięki temu możliwe było szybsze testowanie kolejnych konfiguracji i bardziej iteracyjne strojenie modelu.

Przykładowy wynik wygenerowany przez LSTM:

<audio controls>
  <source src="../../output/example/Generated_DOOM_V78_v7.wav" type="audio/mp4">
  Twoja przeglądarka nie obsługuje odtwarzacza audio.
</audio>

### 8.2 Model transformerowy

Model transformerowy został wytrenowany przez 60 epok, które trenowane były przez 20 minut. W trakcie treningu wartość `train loss` malała systematycznie z 3.655 w epoce 0 do 0.378 w epoce 59, co wskazuje, że model coraz lepiej dopasowywał się do danych treningowych.

Najlepszy wynik walidacyjny uzyskano bardzo wcześnie, już w epoce 2, dla której `val loss` wyniósł 3.391. W kolejnych epokach wartość błędu walidacyjnego stopniowo rosła, osiągając 4.961 w epoce 59. Taki przebieg wskazuje na przeuczenie modelu, czyli poprawę dopasowania do zbioru treningowego kosztem zdolności generalizacji.

Jednocześnie odsłuch wygenerowanych przykładów pokazał, że późniejsze checkpointy, mimo gorszych wartości `val loss`, potrafiły generować utwory brzmiące bardziej naturalne. Oznacza to, że w zadaniu generowania muzyki symbolicznej klasyczna metryka oparta na przewidywaniu kolejnego tokenu nie zawsze jest w pełni zgodna z subiektywną oceną jakości wygenerowanego materiału.

Ocena jakościowa wygenerowanych przykładów pokazała, że jedynie część utworów można było uznać za "przyjemną dla ucha". Około 20% wygenerowanych sekwencji zachowywało względną spójność stylistyczną i rytmiczną, natomiast pozostałe przykłady często miały charakter chaotyczny, złożony z przypadkowych lub słabo powiązanych brzmień.

## 9. Wnioski

Porównanie obu modeli pokazuje, że transformer lepiej radził sobie z modelowaniem zależności w dłuższych sekwencjach, choć rzadzień generował spójny materiał muzyczny niż LSTM. Jednocześnie analiza przebiegu treningu wykazała, że najlepszy wynik walidacyjny został osiągnięty bardzo wcześnie, a dalsze uczenie prowadziło do stopniowego przeuczenia modelu.

Wyniki projektu pokazują również, że w zadaniu generowania muzyki symbolicznej klasyczne metryki, takie jak wartość funkcji straty, nie zawsze są w pełni zgodne z oceną odsłuchową. Mimo pogorszenia `val loss` w późniejszych epokach wygenerowane utwory brzmiały bardziej naturalnie i stylistycznie przekonująco niż przykłady z wcześniejszych checkpointów.

Model LSTM okazał się prostszy obliczeniowo i łatwiejszy do strojenia, co stanowiło jego istotną zaletę przy ograniczonym czasie eksperymentów i zasobach sprzętowych. Z drugiej strony jego ograniczona zdolność do utrzymywania długoterminowej spójności powodowała, że generowane utwory częściej traciły strukturę formalną na dłuższych odcinkach.

Ostatecznie można stwierdzić, że transformer miał większy potencjał do generowania muzyki w stylistyce zbliżonej do materiału źródłowego, jednak jakość wyników pozostawała nierówna. Fakt, że tylko część wygenerowanych utworów była muzycznie satysfakcjonująca, pokazuje, że problem generowania spójnej muzyki symbolicznej nadal pozostaje trudny, szczególnie przy stosunkowo niewielkim zbiorze danych.

W dalszym rozwoju projektu zasadne byłoby zwiększenie zbioru danych, dopracowanie sposobu tokenizacji oraz rozszerzenie oceny modeli o bardziej uporządkowaną analizę jakościową i ilościową. Pozwoliłoby to lepiej porównać modele nie tylko pod względem błędu predykcji, ale również rzeczywistej jakości generowanej muzyki.






Porównanie obu modeli pokazuje, że transformer lepiej radził sobie z modelowaniem zależności w dłuższych sekwencjach i częściej generował bardziej spójny materiał muzyczny. Z drugiej strony LSTM był prostszy obliczeniowo i łatwiejszy do wielokrotnego strojenia, co przy ograniczonym czasie i zasobach sprzętowych stanowi istotną zaletę.

Ostatecznie można stwierdzić, że w tym projekcie transformer okazał się silniejszym modelem pod względem jakości reprezentacji sekwencji, natomiast LSTM pozostawał konkurencyjny jako rozwiązanie lżejsze i szybsze w eksperymentowaniu. W zadaniach o ograniczonym budżecie obliczeniowym LSTM może być więc bardziej praktycznym wyborem, natomiast przy większych zasobach i dłuższym czasie strojenia większy potencjał rozwojowy ma architektura transformerowa.